# Probe dan flash — SWD, RTT, dan DFU

STM32F411 simulasi (`VirtualMcu`) berada di balik probe CMSIS-DAP virtual. Kita membaca memorinya, mem-flash firmware baru, membaca log RTT-nya dan mengirim perintah, lalu memperbaruinya lagi lewat bootloader USB DFU-nya. Dengan board sungguhan pakai `DebugProbe.Create(o => o.UseTarget("STM32F411CEUx"))` (probe-rs) dan `DfuClient.Create(o => o.UseDevice(0x0483, 0xDF11))`.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.29.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.29.0-preview.1"

## Image firmware
Intel HEX, file DFU, dan ELF dimuat ke satu model: segmen, entry point, CRC.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Debug;
using IoTCom.Net.Transport.Usb;

var image = VirtualMcu.BuildDemoFirmware("1.1.0");
Console.WriteLine($"{image}, version tag {image.FindTag("FWVER:")}");
var hex = IntelHex.Format(image);
Console.WriteLine(string.Join("\n", hex.Split('\n').Take(3)));
var dfuFile = DfuFile.Parse(DfuFile.BuildDfuSe(image, 0x0483, 0xDF11));
Console.WriteLine(dfuFile.Describe());

## Probe hanya-baca
Baca memori dan state core berjalan saat firmware jalan; penulisan ditolak.

In [ ]:
var mcu = new VirtualMcu(firmware: VirtualMcu.BuildDemoFirmware("1.0.0"));
mcu.Step(1.5);
var viewer = DebugProbe.Create(o => o.UseVirtual(new VirtualProbeBackend(mcu)).UseTarget("STM32F411CEUx"));
Console.WriteLine($"core {await viewer.CoreAsync()}, uptime {await viewer.ReadUInt32Async(VirtualMcu.UptimeAddress)} ms");
var rtt = await viewer.AttachRttAsync(VirtualMcu.RamBase, VirtualMcu.RamSize);
Console.Write(await rtt.ReadTextAsync());
try { await viewer.FlashAsync(image); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }
await viewer.DisposeAsync();

## Flash lewat SWD, bicara lewat RTT
Hapus, program, verifikasi, reset — lalu firmware baru booting dan menjawab di kanal down.

In [ ]:
var engineer = DebugProbe.Create(o => o.UseVirtual(new VirtualProbeBackend(mcu)).UseTarget("STM32F411CEUx").AllowWrites());
await engineer.FlashAsync(image, new Progress<FlashProgress>(p => { if (p.Fraction >= 1) Console.WriteLine($"{p.Phase} done"); }));
mcu.Step(0.5);
var terminal = await engineer.AttachRttAsync(VirtualMcu.RamBase, VirtualMcu.RamSize);
Console.Write(await terminal.ReadTextAsync());
await terminal.WriteAsync(System.Text.Encoding.ASCII.GetBytes("led on\n"));
mcu.Step(0.1);
Console.Write(await terminal.ReadTextAsync());
Console.WriteLine($"LED {(mcu.Led ? "on" : "off")}");
await engineer.DisposeAsync();

## Pembaruan lewat USB DFU
Chip yang sama di bootloader DfuSe-nya: tata letak memori datang dari string antarmuka; hapus halaman, set alamat, blok, leave.

In [ ]:
var bus = new VirtualUsbBus();
var bootloader = bus.Add(new VirtualDfuDevice(mcu));
var dfu = DfuClient.Create(o => o.UseVirtual(bus).AllowWrites());
await dfu.ConnectAsync();
Console.WriteLine(dfu.Layout!.Describe());
await dfu.DownloadAsync(VirtualMcu.BuildDemoFirmware("2.0.0"));
await dfu.LeaveAsync();
Console.WriteLine($"{bootloader.PagesErased} pages erased, {bootloader.BlocksWritten} blocks; now running {mcu.FirmwareVersion}");
await dfu.DisposeAsync();

## Lebih lanjut
`iotcom probe list`, `iotcom probe flash <file> --target <chip> --allow-write`, `iotcom probe rtt --sim`, `iotcom dfu download --sim --leave --allow-write`, *Probe dan flash firmware* di Gallery, dan sampel ProbeFlash. Lihat `docs/id/protocols/debug.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*